In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import batman
import emcee
import corner

# load K2 data
data = np.loadtxt(
    "../data/apjladf06ft3_mrt.txt",
    skiprows=15
)

t = data[:, 0] + 2454833 #offset to BJD
flux = data[:, 2]

# published mid-transit time from Venner et al. (2026)
t0_published = 2458055.0969

mask = np.abs(t - t0_published) < 0.8 # keep only data points within +/- 0.8 days (19.2h) around the transit

t_transit = t[mask]
flux_transit = flux[mask]

In [ ]:
# Gravitational constant in SI units
G = 6.67430e-11

# the paper finds an out-of-transit scatter of about 32 ppm
# for the K2 light curve.
sigma_flux = 32e-6

# Stellar-density prior from Venner et al. (2026):
# log10(rho_star / (g cm^-3)) = 0.462 +/- 0.041
logrho_mean = 0.462
logrho_sigma = 0.041

# Limb-darkening priors in the Kepler band
u1_mean = 0.60
u1_sigma = 0.07

u2_mean = 0.12
u2_sigma = 0.07

In [ ]:
def a_over_rstar(period, rho_star):
    """
    Calculate a/R_star using Kepler's third law.

    period   : orbital period in days
    rho_star : stellar density in g/cm^3
    """

    # days -> seconds
    period_seconds = period * 24 * 60 * 60

    # g/cm^3 -> kg/m^3
    rho_si = rho_star * 1000

    return (
        G * rho_si * period_seconds**2
        / (3 * np.pi)
    )**(1/3)

print(a_over_rstar(355, 2.90))

In [ ]:
# Starting parameter object only used to initialise batman
batman_params = batman.TransitParams()

batman_params.t0 = t0_published
batman_params.per = 355.0
batman_params.rp = 0.01368
batman_params.a = 270.0
batman_params.inc = 90.0

batman_params.ecc = 0.0
batman_params.w = 90.0

batman_params.u = [0.60, 0.12]
batman_params.limb_dark = "quadratic"

# The K2 exposure time was 29.4 minutes.
# The paper supersamples each exposure by a factor of 10.
batman_model = batman.TransitModel(
    batman_params,
    t_transit,
    supersample_factor=10,
    exp_time=29.4 / (24 * 60)
)

In [ ]:
def transit_model(theta):
    """
    Circular transit model.

    theta =
    (period, b, rp, t0, rho_star, u1, u2)
    """

    period, b, rp, t0, rho_star, u1, u2 = theta

    # Calculate a/Rstar from period and stellar density
    a_Rstar = a_over_rstar(period, rho_star)

    # For a circular orbit:
    # b = (a/Rstar) * cos(i)
    cos_i = b / a_Rstar

    # Reject geometrically impossible solutions
    if cos_i < 0 or cos_i > 1:
        return None

    inclination = np.degrees(np.arccos(cos_i))

    # Update the batman parameters
    batman_params.t0 = t0
    batman_params.per = period
    batman_params.rp = rp
    batman_params.a = a_Rstar
    batman_params.inc = inclination

    # Circular orbit for our baseline model
    batman_params.ecc = 0.0
    batman_params.w = 90.0

    batman_params.u = [u1, u2]

    return batman_model.light_curve(batman_params)

In [ ]:
def log_prior(theta):
    """
    Priors for the circular baseline model.

    We try to follow the assumptions of Venner et al. (2026)
    more closely here.
    """

    period, b, rp, t0, rho_star, u1, u2 = theta


    # ----- Basic physical limits -----

    # No second transit was seen during K2, so the period
    # must be longer than about 64 days.
    # Venner et al. use 2000 d as an upper limit.
    if not 64 < period < 2000:
        return -np.inf

    if not 0 <= b < 1:
        return -np.inf

    if not 0.005 < rp < 0.03:
        return -np.inf

    if not t0_published - 0.1 < t0 < t0_published + 0.1:
        return -np.inf

    if rho_star <= 0:
        return -np.inf


    # Physical conditions for quadratic limb darkening
    if not (
        u1 > 0
        and u1 + u2 < 1
        and u1 + 2*u2 > 0
    ):
        return -np.inf


    lp = 0.0


    # ----- Single-transit period prior -----
    #
    # Long-period planets are less likely to have happened
    # to transit during the observing window.
    #
    # Venner et al. use the Kipping (2018) single-transit prior,
    # which scales approximately as P^(-5/3).

    lp += -(5/3) * np.log(period)


    # ----- Stellar density prior -----
    #
    # Venner et al. use:
    # log10(rho_star) = 0.462 +/- 0.041

    log_rho = np.log10(rho_star)

    lp += -0.5 * (
        (log_rho - 0.462) / 0.041
    )**2


    # ----- Limb-darkening priors -----

    lp += -0.5 * ((u1 - 0.60) / 0.07)**2
    lp += -0.5 * ((u2 - 0.12) / 0.07)**2


    # Radius ratio was fitted approximately log-uniformly,
    # so add the corresponding 1/rp prior.
    lp += -np.log(rp)


    return lp

In [ ]:
#prior

def log_prior(theta):
    """
    Priors for the circular baseline model.

    We try to follow Venner et al. (2026) as closely as
    possible from the description in the paper.
    """

    period, b, rp, t0, rho_star, u1, u2 = theta


    # --------------------------------------------------
    # Basic allowed parameter ranges
    # --------------------------------------------------

    # No second K2 transit was seen, giving P > ~64 d.
    # The paper sets an upper limit of 2000 d.
    if not 64 < period < 2000:
        return -np.inf

    # A transit is possible up to approximately b = 1 + Rp/Rstar.
    if not 0 <= b < 1 + rp:
        return -np.inf

    if not 0.005 < rp < 0.03:
        return -np.inf

    # We already know quite accurately where the transit occurred.
    if not t0_published - 0.1 < t0 < t0_published + 0.1:
        return -np.inf

    if rho_star <= 0:
        return -np.inf


    # --------------------------------------------------
    # Physical quadratic limb-darkening conditions
    # --------------------------------------------------

    if not (
        u1 > 0
        and u1 + u2 < 1
        and u1 + 2*u2 > 0
    ):
        return -np.inf


    # Check that the geometry itself is possible
    a_Rstar = a_over_rstar(period, rho_star)

    if b > a_Rstar:
        return -np.inf


    lp = 0.0


    # --------------------------------------------------
    # Period prior
    # --------------------------------------------------

    # The paper fits P log-uniformly...
    lp += -np.log(period)

    # ...and additionally applies the Kipping single-transit
    # probability prior. The paper states that its long-period
    # behaviour scales approximately as P^(-5/3).
    #
    # This is our approximation of that prior from the
    # description in the paper.
    lp += -(5/3) * np.log(period)


    # --------------------------------------------------
    # Radius-ratio prior
    # --------------------------------------------------

    # Rp/Rstar is fitted log-uniformly in the paper
    lp += -np.log(rp)


    # --------------------------------------------------
    # Stellar-density prior
    # --------------------------------------------------

    # rho_star is fitted log-uniformly
    lp += -np.log(rho_star)

    # plus the stellar constraint:
    # log10(rho_star) = 0.462 +/- 0.041
    log_rho = np.log10(rho_star)

    lp += -0.5 * (
        (log_rho - logrho_mean) / logrho_sigma
    )**2


    # --------------------------------------------------
    # Limb-darkening priors
    # --------------------------------------------------

    lp += -0.5 * ((u1 - u1_mean) / u1_sigma)**2
    lp += -0.5 * ((u2 - u2_mean) / u2_sigma)**2


    return lp

In [ ]:
#likelihood

def log_likelihood(theta):
    """
    Gaussian likelihood comparing the observed flux
    with the transit model.
    """

    model_flux = transit_model(theta)

    # Invalid transit geometry
    if model_flux is None:
        return -np.inf

    if not np.all(np.isfinite(model_flux)):
        return -np.inf

    residuals = flux_transit - model_flux

    return -0.5 * np.sum(
        residuals**2 / sigma_flux**2
        + np.log(2 * np.pi * sigma_flux**2)
    )

In [ ]:
#posterior

def log_probability(theta):
    """
    log posterior = log prior + log likelihood
    """

    lp = log_prior(theta)

    if not np.isfinite(lp):
        return -np.inf

    return lp + log_likelihood(theta)

In [ ]:
#quick test
theta_test = (
    355.0,          # period [days]
    0.3,            # impact parameter
    0.01368,        # Rp/Rstar
    t0_published,   # mid-transit time
    2.90,           # stellar density [g/cm^3]
    0.60,           # u1
    0.12            # u2
)

print("log prior:      ", log_prior(theta_test))
print("log likelihood: ", log_likelihood(theta_test))
print("log posterior:  ", log_probability(theta_test))

In [ ]:
model_test = transit_model(theta_test)

plt.figure(figsize=(9, 5))

plt.plot(
    (t_transit - t0_published) * 24,
    flux_transit,
    ".",
    label="K2 data"
)

plt.plot(
    (t_transit - t0_published) * 24,
    model_test,
    label="Circular transit model"
)

plt.xlabel("Time from mid-transit [hours]")
plt.ylabel("Detrended flux")
plt.legend()
plt.show()

In [ ]:
#mcmc
ndim = 7
nwalkers = 50

labels = [
    "Period [d]",
    "b",
    "Rp/Rstar",
    "t0 [BJD]",
    "rho_star",
    "u1",
    "u2"
]

initial = np.array([
    355.0,
    0.3,
    0.01368,
    t0_published,
    2.90,
    0.60,
    0.12
])

spread = np.array([
    30.0,      # period
    0.15,      # b
    0.0005,    # Rp/Rstar
    0.005,     # t0
    0.10,      # rho_star
    0.03,      # u1
    0.03       # u2
])

rng = np.random.default_rng(42)

In [ ]:
acceptance = sampler.acceptance_fraction

print("Mean acceptance fraction:")
print(np.mean(acceptance))

In [ ]:
samples = sampler.get_chain()

labels = [
    "Period [d]",
    "b",
    "Rp/Rstar",
    "t0 [BJD]",
    "rho_star",
    "u1",
    "u2"
]

fig, axes = plt.subplots(
    len(labels),
    figsize=(10, 12),
    sharex=True
)

for i in range(len(labels)):
    axes[i].plot(samples[:, :, i], alpha=0.25)
    axes[i].set_ylabel(labels[i])

axes[-1].set_xlabel("MCMC step")

plt.tight_layout()
plt.show()

In [ ]:
pos = []

while len(pos) < nwalkers:

    candidate = initial + spread * rng.normal(size=ndim)

    if np.isfinite(log_probability(candidate)):
        pos.append(candidate)

pos = np.array(pos)

print(pos.shape)

In [ ]:
#first check

sampler = emcee.EnsembleSampler(
    nwalkers,
    ndim,
    log_probability
)

sampler.run_mcmc(
    pos,
    10000,
    progress=True
)

In [ ]:
print(sampler.get_chain().shape)

In [ ]:
#diagonstics
print(
    "Mean acceptance fraction:",
    np.mean(sampler.acceptance_fraction)
)

tau = sampler.get_autocorr_time(tol=0)

for label, value in zip(labels, tau):
    print(f"{label}: tau = {value:.1f}")

print()
print("Chain length / tau:")

for label, value in zip(labels, tau):
    print(
        f"{label}: "
        f"{sampler.iteration / value:.1f}"
    )

In [ ]:
chain = sampler.get_chain()

fig, axes = plt.subplots(
    ndim,
    figsize=(10, 13),
    sharex=True
)

for i in range(ndim):

    axes[i].plot(
        chain[:, :, i],
        alpha=0.25
    )

    axes[i].set_ylabel(labels[i])

axes[-1].set_xlabel("MCMC step")

plt.tight_layout()
plt.show()

In [ ]:
#stability check

chain = sampler.get_chain()

N = sampler.iteration

# Compare two later sections of the chain.
# We deliberately ignore the first quarter.
start = N // 4
middle = (start + N) // 2

samples_earlier = chain[start:middle].reshape(-1, ndim)
samples_later = chain[middle:N].reshape(-1, ndim)

for i, label in enumerate(labels):

    q_earlier = np.percentile(
        samples_earlier[:, i],
        [16, 50, 84]
    )

    q_later = np.percentile(
        samples_later[:, i],
        [16, 50, 84]
    )

    print(label)
    print("  earlier:", q_earlier)
    print("  later:  ", q_later)

In [ ]:
#posterior samples

burnin = sampler.iteration // 2
thin = 100

flat_samples = sampler.get_chain(
    discard=burnin,
    thin=thin,
    flat=True
)

print("Total chain length:", sampler.iteration)
print("Burn-in:", burnin)
print("Posterior samples:", len(flat_samples))

In [ ]:
#corner plot

published_values = [
    355.0,
    None,           # paper does not give a median b
    0.01368,
    2458055.0969,
    2.90,
    0.56,
    0.10
]

fig = corner.corner(
    flat_samples,
    labels=labels,
    show_titles=True,
    quantiles=[0.16, 0.50, 0.84],
    truths=published_values
)

plt.show()

In [ ]:
#numerical posterior summary

for i, label in enumerate(labels):

    q16, q50, q84 = np.percentile(
        flat_samples[:, i],
        [16, 50, 84]
    )

    print(
        f"{label}: "
        f"{q50:.5f} "
        f"-{q50-q16:.5f} "
        f"+{q84-q50:.5f}"
    )

In [ ]:
#posterior predictive check

rng = np.random.default_rng(1)

n_models = min(100, len(flat_samples))

indices = rng.choice(
    len(flat_samples),
    size=n_models,
    replace=False
)

plt.figure(figsize=(9, 5))

# Plot posterior transit models first
for index in indices:

    model_flux = transit_model(
        flat_samples[index]
    )

    plt.plot(
        (t_transit - t0_published) * 24,
        model_flux,
        alpha=0.05
    )

# Observations on top
plt.plot(
    (t_transit - t0_published) * 24,
    flux_transit,
    ".",
    label="K2 data"
)

plt.xlabel("Time from mid-transit [hours]")
plt.ylabel("Detrended flux")
plt.title("Circular-model posterior predictive check")
plt.legend()

plt.show()

In [ ]:
#running another 10k steps

sampler.run_mcmc(
    None,
    10000,
    progress=True
)

In [ ]:
tau = sampler.get_autocorr_time(tol=0)

for label, value in zip(labels, tau):
    print(
        f"{label}: tau = {value:.1f}, "
        f"N/tau = {sampler.iteration/value:.1f}"
    )

In [ ]:
np.save(
    "../data/circular_posterior.npy",
    flat_samples
)

In [ ]:
burnin = sampler.iteration // 2
thin = 100

flat_samples = sampler.get_chain(
    discard=burnin,
    thin=thin,
    flat=True
)

print(np.percentile(flat_samples[:, 0], [16, 50, 84]))

In [ ]:
np.save(
    "../data/posterior_circular.npy",
    flat_samples
)

In [ ]:
# Comparison with the published posterior
'''
The posterior samples from Venner et al. (2026) were kindly provided by
Alexander Venner. We compare our circular model with their original
posterior as a validation of our implementation.
'''

In [ ]:
import pandas as pd

venner = pd.read_csv(
    "../data/posteriors.csv"
)

venner.head()

In [ ]:
period_venner = venner["P"].to_numpy()

print(
    "Venner posterior:",
    np.percentile(
        period_venner,
        [16, 50, 84]
    )
)

In [ ]:
period_ours = flat_samples[:, 0]
bins = np.linspace(
    150,
    1000,
    80
)

plt.figure(figsize=(9, 5))

plt.hist(
    period_venner,
    bins=bins,
    density=True,
    histtype="step",
    linewidth=2,
    label="Venner et al."
)

plt.hist(
    period_ours,
    bins=bins,
    density=True,
    histtype="step",
    linewidth=2,
    label="Our circular model"
)

plt.xlabel("Orbital period [days]")
plt.ylabel("Posterior density")
plt.title("Validation of the circular-orbit model")

plt.legend()
plt.show()